In [0]:
from pyspark.sql.functions import col

In [0]:
# 1. Caminhos de Leitura (Silver) e Escrita (Gold)
source_silver_sia = "oftalmo_sus.02_silver.tb_fato_sia"
source_silver_sih = "oftalmo_sus.02_silver.tb_fato_sih"
source_silver_sigtap = "oftalmo_sus.02_silver.tb_dim_sigtap_oftalmo" # Dicionário filtrado
source_silver_ibge = "oftalmo_sus.02_silver.tb_dim_municipios_ibge" # Municípios

target_gold_ambulatorio = "oftalmo_sus.03_gold.mart_ambulatorio_oftalmo"
target_gold_hospitalar = "oftalmo_sus.03_gold.mart_hospitalar_oftalmo"

dbutils.widgets.text("year", "2025", "Year (YYYY)")
dbutils.widgets.text("month", "01", "Month (MM)")
dbutils.widgets.text("debug", "False", "Debug Mode")

year = dbutils.widgets.get("year")
month = dbutils.widgets.get("month")
debug = dbutils.widgets.get("debug")

if debug == "False":
  debug = False
else:
  debug = True

print(f"Month: {month}, Year: {year}, Debug Mode: {debug}")

### 🔄 Processando e enriquecendo dados Ambulatoriais (SIA)

In [0]:
# 2. Carregando as Tabelas Silver
df_sia = spark.read.table(source_silver_sia).filter(col("batch_processing") == f"{year}{month}")
df_sih = spark.read.table(source_silver_sih).filter(col("batch_processing") == f"{year}{month}")
df_sigtap = spark.read.table(source_silver_sigtap)
df_ibge = spark.read.table(source_silver_ibge)

In [0]:
print("🔄 Processando e enriquecendo dados Ambulatoriais (SIA)...")

# Join com SIGTAP utilizando a conversão matemática validada (ignora zeros à esquerda/fantasmas)
df_gold_sia = df_sia.join(
    df_sigtap,
    col("PA_PROC_ID").cast("bigint") == col("CO_PROCEDIMENTO").cast("bigint"),
    "inner"
)

# Join com a tabela do IBGE mapeando o código de 6 dígitos do município do paciente
df_gold_sia_geo = df_gold_sia.join(
    df_ibge,
    col("PA_MUNPCN").cast("bigint") == col("codigo_ibge_6").cast("bigint"),
    "left"
)

In [0]:
# Escrita incremental na tabela Delta Gold de Ambulatório
df_gold_sia_geo.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true")\
    .partitionBy("batch_processing") \
    .saveAsTable(target_gold_ambulatorio)

total_sia_gold = df_gold_sia_geo.count()
print(f"✅ Data Mart Ambulatorial atualizado! Registros adicionados: {total_sia_gold:,}")

### 🔄 Processando e enriquecendo dados Hospitalares (SIH)

In [0]:
# Espelhamento da arquitetura de proteção de tipos para o código cirúrgico do SIH
df_gold_sih = df_sih.join(
    df_sigtap,
    col("SP_PROCREA").cast("bigint") == col("CO_PROCEDIMENTO").cast("bigint"),
    "inner"
)

# Join com o IBGE utilizando o campo de município do paciente do SIH (SP_MUNIC)
df_gold_sih_geo = df_gold_sih.join(
    df_ibge,
    col("SP_M_PAC").cast("bigint") == col("codigo_ibge_6").cast("bigint"),
    "left"
)

In [0]:
display(df_gold_sih_geo.limit(10))

In [0]:
# Escrita incremental na tabela Delta Gold de Internações/Cirurgias
df_gold_sih_geo.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true")\
    .partitionBy("batch_processing") \
    .saveAsTable(target_gold_hospitalar)

total_sih_gold = df_gold_sih_geo.count()
print(f"✅ Data Mart Hospitalar atualizado! Registros adicionados: {total_sih_gold:,}")